# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khaled-dragon/ML-intern/blob/main/work/notebooks/w04_signal_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
%pip -q install duckdb huggingface_hub
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

Paste your HF READ token (hf_...): ··········


In [3]:
dist_data = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']} WHERE month='2026-03'),
    prior AS (
        SELECT content_hash_id,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY THEN gsc_clicks ELSE 0 END) AS clk_prev30,
               AVG(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY THEN gsc_avg_position END) AS pos_prev30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.month = '2026-03'
        GROUP BY 1
        HAVING imp_prev30 > 0
    )
    SELECT * FROM prior
""").df()

print(dist_data[['imp_prev30', 'clk_prev30', 'pos_prev30']].describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

          imp_prev30     clk_prev30     pos_prev30
count  176268.000000  176268.000000  176268.000000
mean     1548.696377       4.525308      16.073409
std      5305.808802      26.153943      17.774402
min         1.000000       0.000000       0.000000
25%        19.000000       0.000000       5.004870
50%       170.000000       0.000000       8.535847
75%      1014.000000       2.000000      20.505243
max    613219.000000    5631.000000     309.000000


Both imp_prev30 and clk_prev30 show heavy right tails: the mean (1,548
impressions, 4.5 clicks) sits far above the median (170 impressions, 0
clicks), meaning a small number of high-traffic pages pull the average up
while most pages sit much lower. Half of all pages had zero clicks in the
prior 30 days. pos_prev30 ranges from 0 to 309 with a median around 8.5,
confirming the earlier finding that "position zero" pages are a real,
present-in-force feature of this data, not an isolated few.

### "#########################################################################"

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [4]:
signal3 = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']} WHERE month='2026-03'),
    prior AS (
        SELECT content_hash_id,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY THEN gsc_clicks ELSE 0 END) AS clk_prev30,
               AVG(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY THEN gsc_avg_position END) AS pos_prev30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.month = '2026-03'
        GROUP BY 1
        HAVING imp_prev30 >= 100
    )
    SELECT
        CASE WHEN pos_prev30 <= 10 THEN 'top_10'
             WHEN pos_prev30 <= 30 THEN 'top_30'
             ELSE 'beyond_30' END AS position_bucket,
        COUNT(*) AS n,
        AVG(clk_prev30 * 1.0 / NULLIF(imp_prev30, 0)) AS avg_ctr
    FROM prior
    GROUP BY 1
    ORDER BY 2 DESC
""").df()
print(signal3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_bucket      n   avg_ctr
0          top_10  55402  0.003289
1          top_30  32801  0.002128
2       beyond_30  12646  0.000855


**Signal 3 — CTR vs position (linked to the CTR-fix flag): CONFIRMED.**
Average CTR drops sharply and consistently as position gets worse: 0.33% in
top_10, down to 0.21% in top_30, down to 0.09% beyond_30 — roughly a 4x drop
from best to worst position bucket. This is a clean, monotonic pattern with
large sample sizes in every bucket (12,646 to 55,402 rows), so unlike the
volume and position-vs-growth signals, this one holds up well and is safe
to use as a scoring input.

### "#########################################################################"

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [5]:
ctr_outliers = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']} WHERE month='2026-03'),
    prior AS (
        SELECT content_hash_id,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY THEN gsc_clicks ELSE 0 END) AS clk_prev30,
               AVG(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY THEN gsc_avg_position END) AS pos_prev30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.month = '2026-03'
        GROUP BY 1
        HAVING imp_prev30 >= 100
    )
    SELECT COUNT(*) AS underperforming_pages
    FROM prior
    WHERE pos_prev30 <= 10
      AND (clk_prev30 * 1.0 / imp_prev30) < 0.003289 * 0.5
""").df()
print(ctr_outliers)

   underperforming_pages
0                  24881


Of the 55,402 pages in the top_10 position bucket, 24,881 (about 45%) have
a CTR below half the bucket's average, despite ranking well. That's a large
enough group to be a real, actionable pattern, not noise: nearly half of the
best-ranked pages are underperforming on clicks relative to their peers in
the same position tier. This is exactly the kind of gap a CTR-fix flag
exists to catch: good position, weak title/snippet appeal.

### "#########################################################################"

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

For a content team with limited review time, three things matter here:
first, don't trust raw traffic volume as a "quick win" signal, it's actually
inversely related to growth in this data. Second, position alone doesn't
predict growth either, it needs to be paired with other signals. Third, the
CTR-vs-position pattern is the most reliable one found here: the ~45% of
top-10 pages with below-average CTR are a concrete, sizeable list worth a
metadata/snippet review before touching content itself, since the ranking
is already good and the fix is likely cheaper than a content rewrite.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.